# Project 3: Cart-Pole Trajectory Optimization

基于 MIT underactuated.mit.edu 官方代码 `dircol.ipynb` 完成
运行环境：VMware + Ubuntu 22.04 + VS Code Jupyter + Drake

In [ ]:
# Part 1: 导入库

import numpy as np
from IPython.display import HTML, display
from matplotlib import pyplot as plt

from pydrake.all import (
    ConnectPlanarSceneGraphVisualizer,
    DiagramBuilder,
    DirectCollocation,
    FiniteHorizonLinearQuadraticRegulator,
    FiniteHorizonLinearQuadraticRegulatorOptions,
    LogVectorOutput,
    MakeFiniteHorizonLinearQuadraticRegulator,
    MultibodyPlant,
    Parser,
    PiecewisePolynomial,
    PlanarSceneGraphVisualizer,
    RobotDiagramBuilder,
    SceneGraph,
    Simulator,
    Solve,
    TrajectorySource,
)
from pydrake.examples import (
    AcrobotGeometry, AcrobotPlant, PendulumPlant, PendulumState,
)

from underactuated import ConfigureParser, running_as_notebook
from underactuated.pendulum import PendulumVisualizer
from underactuated.pyplot_utils import (
    AdvanceToAndVisualize, AnimatePositionTrajectory,
)



# 修复VS Code Jupyter兼容性：running_as_notebook检测失败
running_as_notebook = True



print('[OK] 导入完成')


In [ ]:
# Part 2: 直接梯形配点法优化
# 求解Cart-Pole swing-up最优轨迹

def dircol_cartpole():
    builder = RobotDiagramBuilder(time_step=0.0)
    ConfigureParser(builder.parser())
    builder.parser().AddModelsFromUrl(
        'package://underactuated/models/cartpole.urdf')
    diagram = builder.Build()
    context = diagram.CreateDefaultContext()
    dircol = DirectCollocation(
        diagram, context, num_time_samples=21,
        minimum_time_step=0.1, maximum_time_step=0.4,
        input_port_index=diagram.GetInputPort(
            'plant_actuation').get_index())
    prog = dircol.prog()
    dircol.AddEqualTimeIntervalsConstraints()
    initial_state = (0.0, 0.0, 0.0, 0.0)
    prog.AddBoundingBoxConstraint(
        initial_state, initial_state, dircol.initial_state())
    final_state = (0.0, np.pi, 0.0, 0.0)
    prog.AddBoundingBoxConstraint(
        final_state, final_state, dircol.final_state())
    R = 10
    u = dircol.input()
    dircol.AddRunningCost(R * u[0] ** 2)
    dircol.AddFinalCost(dircol.time())
    initial_x_trajectory = PiecewisePolynomial.FirstOrderHold(
        [0.0, 4.0], np.column_stack((initial_state, final_state)))
    dircol.SetInitialTrajectory(
        PiecewisePolynomial(), initial_x_trajectory)
    result = Solve(prog)
    assert result.is_success()
    u_trajectory = dircol.ReconstructInputTrajectory(result)
    x_trajectory = dircol.ReconstructStateTrajectory(result)
    return x_trajectory, u_trajectory, result


print('=' * 60)
print('Part 2: 求解 Cart-Pole Swing-up 轨迹优化')
print('=' * 60)
x_trajectory, u_trajectory, result = dircol_cartpole()


# 提取终止状态
x_f = x_trajectory.value(u_trajectory.end_time())
print(f'优化成功! 总时长: {u_trajectory.end_time():.3f}s')
print(f'终止位置: {float(np.array(x_f).flatten()[0]):.4f} m')
print(f'终止角度: {np.degrees(float(np.array(x_f).flatten()[1])):.2f} deg')
print(f'目标函数: {result.get_optimal_cost():.2f}')

# 绘制最优控制力
fig, ax = plt.subplots(figsize=(10, 4))
times = np.linspace(u_trajectory.start_time(), u_trajectory.end_time(), 100)
u_values = u_trajectory.vector_values(times)
ax.plot(times, u_values.T, 'b-', lw=2)
ax.set_xlabel('time [s]')
ax.set_ylabel('force [N]')
ax.set_title('Optimal Control Force (MIT DirectCollocation)')
ax.grid(True)
plt.show()


In [ ]:
# Part 3: 开环仿真验证
# 使用优化轨迹的控制输入驱动系统，验证动力学

def open_loop_simulation(x_traj, u_traj):
    robot_builder = RobotDiagramBuilder(time_step=0.0)
    builder = robot_builder.builder()
    parser = robot_builder.parser()
    plant = robot_builder.plant()
    scene_graph = robot_builder.scene_graph()
    ConfigureParser(parser)
    parser.AddModelsFromUrl(
        'package://underactuated/models/cartpole.urdf')
    plant.Finalize()
    traj = builder.AddSystem(TrajectorySource(u_traj))
    builder.Connect(
        traj.get_output_port(),
        plant.get_actuation_input_port())
    visualizer = ConnectPlanarSceneGraphVisualizer(
        builder, scene_graph,
        xlim=[-2, 2], ylim=[-1.25, 2], show=False)
    diagram = robot_builder.Build()
    simulator = Simulator(diagram)
    AdvanceToAndVisualize(
        simulator, visualizer, u_traj.end_time())


print('\nPart 3: 开环仿真验证')
open_loop_simulation(x_trajectory, u_trajectory)


In [ ]:
# Part 4: TVLQR 闭环控制 
# 沿最优轨迹线性化，设计时变LQR控制器

def finite_horizon_lqr(x_trajectory, u_trajectory):
    options = FiniteHorizonLinearQuadraticRegulatorOptions()
    options.x0 = x_trajectory
    options.u0 = u_trajectory
    robot_builder = RobotDiagramBuilder(time_step=0.0)
    builder = robot_builder.builder()
    parser = robot_builder.parser()
    plant = robot_builder.plant()
    scene_graph = robot_builder.scene_graph()
    ConfigureParser(parser)
    parser.AddModelsFromUrl(
        'package://underactuated/models/cartpole.urdf')
    plant.Finalize()
    context = plant.CreateDefaultContext()
    Q = np.diag([10.0, 10.0, 1.0, 1.0])
    options.Qf = Q
    options.input_port_index = (
        plant.get_actuation_input_port().get_index())
    regulator = builder.AddSystem(
        MakeFiniteHorizonLinearQuadraticRegulator(
            plant, context,
            t0=options.u0.start_time(),
            tf=options.u0.end_time(),
            Q=Q, R=np.eye(1),
            options=options))
    builder.Connect(
        regulator.get_output_port(0),
        plant.get_actuation_input_port())
    builder.Connect(
        plant.get_state_output_port(),
        regulator.get_input_port(0))
    input_logger = LogVectorOutput(
        regulator.get_output_port(0), builder)
    state_logger = LogVectorOutput(
        plant.get_state_output_port(), builder)
    visualizer = ConnectPlanarSceneGraphVisualizer(
        builder, scene_graph,
        xlim=[-2, 2], ylim=[-1.25, 2], show=False)
    diagram = robot_builder.Build()
    simulator = Simulator(diagram)
    AdvanceToAndVisualize(
        simulator, visualizer, options.u0.end_time())
    


    # 绘制开环vs闭环对比
    fig, ax = plt.subplots(2, 1, figsize=(10, 8))
    ax[0].plot(
        options.u0.get_segment_times(),
        options.u0.vector_values(
            options.u0.get_segment_times()).T,
        'b-', lw=2, label='u nominal')
    input_log = input_logger.FindLog(
        simulator.get_context())
    ax[0].plot(
        input_log.sample_times(),
        input_log.data().T,
        'r--', lw=1.5, label='u actual')
    ax[0].set_title('Control Input Comparison')
    ax[0].set_xlabel('time [s]')
    ax[0].set_ylabel('force [N]')
    ax[0].legend()
    ax[0].grid(True)
    ax[1].plot(
        options.x0.get_segment_times(),
        options.x0.vector_values(
            options.x0.get_segment_times())[1, :].T,
        'b-', lw=2, label='theta nominal')
    state_log = state_logger.FindLog(
        simulator.get_context())
    ax[1].plot(
        state_log.sample_times(),
        state_log.data()[1, :].T,
        'r--', lw=1.5, label='theta actual')
    ax[1].set_title('Pole Angle Comparison')
    ax[1].set_xlabel('time [s]')
    ax[1].set_ylabel('theta [rad]')
    ax[1].legend()
    ax[1].grid(True)
    plt.tight_layout()
    plt.show()


print('\nPart 4: TVLQR闭环控制')
finite_horizon_lqr(x_trajectory, u_trajectory)


In [ ]:
# Part 5: S(t) 代价矩阵分析

def plot_S_trajectory(x_trajectory, u_trajectory):
    options = FiniteHorizonLinearQuadraticRegulatorOptions()
    options.x0 = x_trajectory
    options.u0 = u_trajectory
    plant = MultibodyPlant(time_step=0.0)
    scene_graph = SceneGraph()
    plant.RegisterAsSourceForSceneGraph(scene_graph)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl(
        'package://underactuated/models/cartpole.urdf')
    plant.Finalize()
    context = plant.CreateDefaultContext()
    Q = np.diag([10.0, 10.0, 1.0, 1.0])
    options.Qf = Q
    options.input_port_index = (
        plant.get_actuation_input_port().get_index())
    results = FiniteHorizonLinearQuadraticRegulator(
        plant, context,
        t0=options.u0.start_time(),
        tf=options.u0.end_time(),
        Q=Q, R=np.eye(1),
        options=options)
    S_times = results.S.get_segment_times()
    lambda_max_S = [
        float(np.amax(np.linalg.eigvals(results.S.value(t))))
        for t in S_times]
    fig, ax = plt.subplots(2, 1, figsize=(10, 8))
    ax[0].plot(S_times, lambda_max_S, 'b-', lw=2)
    ax[0].set_xlabel('time [s]')
    ax[0].set_ylabel('lambda_max(S(t))')
    ax[0].set_title('Maximum Eigenvalue of S(t)')
    ax[0].grid(True)
    x_times = options.x0.get_segment_times()
    theta = options.x0.vector_values(x_times)[1, :].T
    ax[1].plot(x_times, theta, 'b-', lw=2, label='theta(t)')
    ax[1].plot([x_times[0], x_times[-1]],
               [np.pi/2, np.pi/2],
               'r--', lw=1, label='horizontal')
    ax[1].set_xlabel('time [s]')
    ax[1].set_ylabel('theta [rad]')
    ax[1].set_title('Optimal Pole Angle')
    ax[1].legend()
    ax[1].grid(True)
    plt.tight_layout()
    plt.show()
    return lambda_max_S


print('\nPart 5: S(t) 代价矩阵分析')
lambda_max_S = plot_S_trajectory(x_trajectory, u_trajectory)
print(f'S(t)最大特征值范围: [{min(lambda_max_S):.1f}, {max(lambda_max_S):.1f}]')


In [ ]:
# Part 6: 消融实验 
# 鉴于先前约束力20N并不能很好的进行实验，现在取值50N

def dircol_cartpole_constrained(x_max_c=2.0, num_samples=21, u_max=50.0):
    builder = RobotDiagramBuilder(time_step=0.0)
    ConfigureParser(builder.parser())
    builder.parser().AddModelsFromUrl(
        'package://underactuated/models/cartpole.urdf')
    diagram = builder.Build()
    context = diagram.CreateDefaultContext()
    dircol = DirectCollocation(
        diagram, context, num_time_samples=num_samples,
        minimum_time_step=0.1, maximum_time_step=0.4,
        input_port_index=diagram.GetInputPort(
            'plant_actuation').get_index())
    prog = dircol.prog()
    dircol.AddEqualTimeIntervalsConstraints()
    initial_state = (0.0, 0.0, 0.0, 0.0)
    prog.AddBoundingBoxConstraint(
        initial_state, initial_state, dircol.initial_state())
    final_state = (0.0, np.pi, 0.0, 0.0)
    prog.AddBoundingBoxConstraint(
        final_state, final_state, dircol.final_state())
    x_var = dircol.state()
    dircol.AddConstraintToAllKnotPoints(
        -x_max_c <= x_var[0])
    dircol.AddConstraintToAllKnotPoints(
        x_var[0] <= x_max_c)
    u_var = dircol.input()
    dircol.AddConstraintToAllKnotPoints(
        -u_max <= u_var[0])
    dircol.AddConstraintToAllKnotPoints(
        u_var[0] <= u_max)
    R = 10
    dircol.AddRunningCost(R * u_var[0] ** 2)
    dircol.AddFinalCost(dircol.time())
    init_traj = PiecewisePolynomial.FirstOrderHold(
        [0.0, 4.0],
        np.column_stack((initial_state, final_state)))
    dircol.SetInitialTrajectory(
        PiecewisePolynomial(), init_traj)
    result = Solve(prog)
    if not result.is_success():
        print(f'  未收敛 x_max={x_max_c}, u_max={u_max}')
        return None, None, result
    u_traj = dircol.ReconstructInputTrajectory(result)
    x_traj = dircol.ReconstructStateTrajectory(result)
    return x_traj, u_traj, result


print('\nPart 6: 消融实验 - 改变位置硬约束 + 电机转矩限制')
print('=' * 60)
configs = [('宽松', 3.0, 'g'), ('标准', 2.0, 'b'), ('严格', 1.2, 'r')]
ab_results = []
for name, xm, col in configs:
    print(f'\n--- {name}: x_max={xm}m, u_max=50N ---')
    x_t, u_t, res = dircol_cartpole_constrained(x_max_c=xm, u_max=50.0)
    if x_t is not None:
        tc = np.linspace(u_t.start_time(), u_t.end_time(), 100)
        uv = u_t.vector_values(tc)
        tx = np.linspace(x_t.start_time(), x_t.end_time(), 100)
        xv = x_t.vector_values(tx)
        ab_results.append({
            'name': name, 'color': col, 'x_max': xm,
            'tc': tc, 'uv': uv, 'tx': tx, 'xv': xv,
            'u_t': u_t, 'result': res})
        print(f'  时长: {u_t.end_time():.3f}s')
        print(f'  目标函数: {res.get_optimal_cost():.2f}')

if ab_results:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for r in ab_results:
        axes[0,0].plot(r['tx'], r['xv'][0,:],
            color=r['color'], lw=2,
            label=f"{r['name']}({r['x_max']})")
    axes[0,0].set_title('Cart Position')
    axes[0,0].set_xlabel('time[s]')
    axes[0,0].legend()
    axes[0,0].grid(True)
    for r in ab_results:
        axes[0,1].plot(r['tx'], np.degrees(r['xv'][1,:]),
            color=r['color'], lw=2, label=r['name'])
    axes[0,1].set_title('Pole Angle')
    axes[0,1].set_xlabel('time[s]')
    axes[0,1].legend()
    axes[0,1].grid(True)
    for r in ab_results:
        axes[1,0].plot(r['tc'], r['uv'][0,:],
            color=r['color'], lw=2, label=r['name'])

    axes[1,0].axhline(y=50, color='k', linestyle='--', alpha=0.3)
    axes[1,0].axhline(y=-50, color='k', linestyle='--', alpha=0.3)
    axes[1,0].set_title('Control Force (|u| <= 50N)')
    axes[1,0].set_xlabel('time[s]')
    axes[1,0].legend()
    axes[1,0].grid(True)
    for r in ab_results:
        axes[1,1].plot(r['xv'][0,:],
            np.degrees(r['xv'][1,:]),
            color=r['color'], lw=2, label=r['name'])
    axes[1,1].set_title('Phase Portrait')
    axes[1,1].set_xlabel('x[m]')
    axes[1,1].set_ylabel('theta[deg]')
    axes[1,1].legend()
    axes[1,1].grid(True)
    plt.suptitle('Ablation: Position + Torque Constraint',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('\n' + '='*60)
    print(f'{"约束":<10} {"x_max":<8} {"时长":<10} {"目标函数":<12} {"最大力":<10}')
    for r in ab_results:
        print(f'{r["name"]:<10} {r["x_max"]:<8.1f} '
              f'{r["u_t"].end_time():<10.3f} '
              f'{r["result"].get_optimal_cost():<12.2f} '
              f'{float(np.max(np.abs(r["uv"]))):<10.2f}')
    print('='*60)